This notebook uses as input, the results made by privgen_example.ipynb
It's job is to create "presentable" figures which compares the impact of the PrivGen method

In [1]:
!pip install ucimlrepo memory-profiler synthcity

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.9/153.9 kB 8.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.5/92.5 kB 9.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.3/100.3 kB 8.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of torchtext to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 430.1/430.1 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [2]:
from synthcity.metrics import Metrics
from synthcity.plugins import Plugins
from synthcity.plugins.core.dataloader import GenericDataLoader
import pandas as pd
import time
import psutil
from memory_profiler import memory_usage
from ucimlrepo import fetch_ucirepo # if you want to use UCI databes data
from memory_profiler import memory_usage


[KeOps] Compiling cuda jit compiler engine ... OK
[pyKeOps] Compiling nvrtc binder for python ... OK


In [ ]:
# uncomment next two lines if you want to connect colab notebook to a GDrive directory (recommended)
# from google.colab import drive
#drive.mount('/content/drive')

# change this to the root folder of PrivGen
# the current path works with GDrive
main_path = "/content/drive/Othercomputers/My Mac/Privgen"

In [ ]:
def compare_results(results1: pd.DataFrame, results2: pd.DataFrame):
    """
    used in eval_5 function
    Compares two results tables and determines which results are better for each metric.
    Handles .gt and .syn metrics by focusing only on .syn for comparisons. Adds actual values.

    Args:
        results1 (pd.DataFrame): The first results table.
        results2 (pd.DataFrame): The second results table.

    Returns:
        pd.DataFrame: A DataFrame summarizing which results are better for each .syn metric.
    """
    comparison = []

    for metric in results1.index:
        # Skip ground truth (.gt) metrics
        if metric.endswith(".gt"):
            continue

        # Ensure the corresponding .gt scores are equal before comparing .syn metrics
        if metric.endswith(".syn"):
            corresponding_gt_metric = metric.replace(".syn", ".gt")
            if (results1.loc[corresponding_gt_metric, "mean"] != results2.loc[corresponding_gt_metric, "mean"]):
                comparison.append([metric, "Error", "Ground truth values differ", None, None])
                continue

        # Determine comparison direction
        direction = results1.loc[metric, "direction"]
        value1 = results1.loc[metric, "mean"]
        value2 = results2.loc[metric, "mean"]

        if direction == "minimize":
            diff = value2 - value1
            better = "results 1" if diff > 0 else "results 2"
        elif direction == "maximize":
            diff = value1 - value2
            better = "results 1" if diff > 0 else "results 2"
        else:
            better = "N/A"

        magnitude = "marginally" if abs(diff) < 0.05 else "by far"
        comparison.append([metric, better, magnitude, value1, value2])

    return pd.DataFrame(comparison, columns=["Metric", "Better Results", "Magnitude", "orig", "privgen"])

In [ ]:
def eval_5(base, source, target_column, sensetive_columns, data_name,
           save_path, file_name=None, task='regression', synthesizer='tvae',
           metrics=['sanity', 'stats', 'performance', 'detection', 'privacy']):

  """
  creates a csv file that compares the synthetic data generated from original data (base) and original data+Privegen (source)
  It applies all available synthsizers on two versions of data and then evaluates them
  """
  available_metrics = {
  'sanity': ['data_mismatch', 'common_rows_proportion', 'nearest_syn_neighbor_distance', 'close_values_probability', 'distant_values_probability'],
  'stats': ['jensenshannon_dist', 'chi_squared_test', 'feature_corr', 'inv_kl_divergence', 'ks_test', 'max_mean_discrepancy', 'wasserstein_dist', 'prdc', 'alpha_precision', 'survival_km_distance'],
  'performance': ['linear_model', 'mlp', 'xgb', 'feat_rank_distance'],
  'detection': ['detection_xgb', 'detection_mlp', 'detection_gmm', 'detection_linear'],
  'privacy': ['delta-presence', 'k-anonymization', 'k-map', 'distinct l-diversity', 'identifiability_score']
  }

  needed_metrics = {k:available_metrics[k] for k in metrics}

  results = []
  for src in [base, source]:

    X = src.copy()
    plugin = Plugins().get(synthesizer)
    loader = GenericDataLoader(X, target_column=target_column,
                                  sensitive_columns=sensetive_columns)
    plugin.fit(loader)
    synth = plugin.generate(len(X), random_state=42).dataframe()


    score = Metrics.evaluate(
    X_gt=base,
    X_syn=synth,
    metrics=needed_metrics,
    task_type=task,
    random_state=42
    )

    results.append(score)

  comparison = compare_results(*results)

  # saving results
  if not file_name:
    file_name = f'{data_name}_{synthesizer}'
  comparison.to_csv(f'{save_path}/{file_name}.csv', index=False)
  return comparison


In [ ]:
# all synthesizers
plugins = Plugins()
plugin_names = plugins.list() # list of all syntheizers available in synthcity library

[2024-11-26T18:48:24.216649+0000][218][CRITICAL] module disabled: /usr/local/lib/python3.10/dist-packages/synthcity/plugins/generic/plugin_goggle.py


In [ ]:
def download_dataset(dataset_name):
    # Map dataset names to UCI ML repository IDs
    dataset_ids = {
    "iris": {"id": 53, "target_column": "target", "sensitive_columns": ["sepal width"]},
    "wine": {"id": 109, "target_column": "target", "sensitive_columns": ["Alcohol", "Malicacid","Ash","Alcalinity_of_ash"]},
    "abalone": {"id": 1, "target_column": "target", "sensitive_columns": ["Sex"]},
    "heart disease": {"id": 45, "target_column": "target", "sensitive_columns": ["age", "sex"]},
    "adult": {"id": 2, "target_column": "target", "sensitive_columns": ["age", "race", "sex"]},
    "car evaluation": {"id": 19, "target_column": "target", "sensitive_columns": ["buying"]},
    "automobile": {"id": 10, "target_column": "target", "sensitive_columns": ["price"]},
    "mushroom": {"id": 73, "target_column": "target", "sensitive_columns": ["odor", "bruises"]},
    "german credit": {"id": 144, "target_column": "target", "sensitive_columns": ["Attribute1","Attribute2"]},
    "dry bean": {"id": 602, "target_column": "target", "sensitive_columns": ["Area","Perimeter","MajorAxisLength"]},
    "bike sharing": {"id": 275, "target_column": "target", "sensitive_columns": []},
    "auto_mpg": {"id": 9, "target_column": "target", "sensitive_columns": ["origin"]},
    "RT-IoT2022": {"id": 942, "target_column": "target", "sensitive_columns": []},
    "EEG Eye State": {"id": 264, "target_column": "target", "sensitive_columns": []},
    "Metro": {"id": 492, "target_column": "target", "sensitive_columns": []}
    }
    
    
    # Fetch the dataset ID
    dataset_id = dataset_ids[dataset_name]['id']
    
    # Fetch the dataset
    dataset = fetch_ucirepo(id=dataset_id)
    X = dataset.data.features
    y = dataset.data.targets
    X['target'] = y
    
    return X, dataset_ids[dataset_name]['target_column'], dataset_ids[dataset_name]['sensitive_columns']




Data fetched successfully, but not saved locally.


### Getting original data and data transformed by PrivGen

In [ ]:
# it's the result of applying privgen on original data
source = pd.read_csv('/content/drive/Othercomputers/My Mac/Privgen/data/abalone_8_decoded_data.csv')

# this download a UCI database, otherwise you should replicate what download_dataset function does for your dataset
base,target_column, sensetive_columns = download_dataset('abalone')

In [ ]:
from glob import glob
files = glob('/content/drive/Othercomputers/My Mac/Privgen/results/*.csv') # where we save the results - current path works is Ok for GDrive
existing = [e.split('/')[-1].split('.')[0] for e in files]

# here we can use abalone dataset because we previously applied privgen to it (privgen_example.ipynb) and the required artifacts are already in data directory
data_name = 'abalone' 

# all synthesizers
plugins = Plugins()
plugin_names = plugins.list()

for synthesizer in plugin_names:
  # we don't need survival data AND "aim" model gives errors and we dont want to re-run what we have already done, so...
  if f'{data_name}_{synthesizer}' not in existing and 'survival' not in synthesizer and synthesizer != 'aim':

    print(f'Running {synthesizer}. . . ')
    try:
      start_time = time.time()
      # Monitor memory usage during execution
      start_memory, peak_memory = memory_usage((eval_5(base, source, target_column, sensetive_columns, data_name,
                save_path='/content/drive/Othercomputers/My Mac/Privgen/results', synthesizer=synthesizer)), retval=False, interval=0.1, timeout=None, max_usage=True)
      end_memory = memory_usage(-1, retval=False)[0]  # Memory after function ends

      end_time = time.time()
      execution_time = end_time - start_time

      # Calculate memory differences
      memory_diff = end_memory - start_memory

      # Performance log
      log_line = (
          f"data_name: {data_name}, "
          f"synthesizer: {synthesizer}, "
          f"execution_time: {execution_time:.6f} seconds, "
          f"Peak Memory Usage: {peak_memory:.2f} MB, "
          f"memory_diff_MB: {memory_diff:.2f}\n"
      )

      with open('/content/drive/Othercomputers/My Mac/Privgen/results/performance_log.txt', "a") as log_file:
          log_file.write(log_line)
    except Exception as e:
      print('='*80)
      print(f"An error  with {synthesizer}: {e}")
      print('='*80)


[2024-11-27T08:16:52.104166+0000][460][CRITICAL] module disabled: /usr/local/lib/python3.10/dist-packages/synthcity/plugins/generic/plugin_goggle.py
[2024-11-27T08:16:52.108698+0000][460][CRITICAL] module disabled: /usr/local/lib/python3.10/dist-packages/synthcity/plugins/generic/plugin_goggle.py
[2024-11-27T08:16:52.138931+0000][460][CRITICAL] module disabled: /usr/local/lib/python3.10/dist-packages/synthcity/plugins/generic/plugin_goggle.py


Running timegan. . . 
An error  with timegan: Invalid data type = generic
Running ctgan. . . 


 32%|███▏      | 649/2000 [05:08<10:41,  2.10it/s]
[2024-11-27T08:28:07.637124+0000][460][CRITICAL] module disabled: /usr/local/lib/python3.10/dist-packages/synthcity/plugins/generic/plugin_goggle.py
 37%|███▋      | 749/2000 [05:55<09:53,  2.11it/s]
[2024-11-27T08:39:54.997716+0000][460][CRITICAL] module disabled: /usr/local/lib/python3.10/dist-packages/synthcity/plugins/generic/plugin_goggle.py


An error  with ctgan: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Running dummy_sampler. . . 


[2024-11-27T08:45:55.412997+0000][460][CRITICAL] module disabled: /usr/local/lib/python3.10/dist-packages/synthcity/plugins/generic/plugin_goggle.py
[2024-11-27T08:51:44.851137+0000][460][CRITICAL] module disabled: /usr/local/lib/python3.10/dist-packages/synthcity/plugins/generic/plugin_goggle.py
[2024-11-27T08:51:44.874982+0000][460][CRITICAL] module disabled: /usr/local/lib/python3.10/dist-packages/synthcity/plugins/generic/plugin_goggle.py


An error  with dummy_sampler: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Running survae. . . 
An error  with survae: Invalid data type = generic
Running dpgan. . . 


 35%|███▍      | 699/2000 [07:58<14:51,  1.46it/s]
[2024-11-27T09:05:48.898898+0000][460][CRITICAL] module disabled: /usr/local/lib/python3.10/dist-packages/synthcity/plugins/generic/plugin_goggle.py
 17%|█▋        | 349/2000 [03:58<18:47,  1.46it/s]
[2024-11-27T09:15:46.357621+0000][460][CRITICAL] module disabled: /usr/local/lib/python3.10/dist-packages/synthcity/plugins/generic/plugin_goggle.py


An error  with dpgan: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Running adsgan. . . 


  5%|▌         | 549/10000 [04:21<1:14:56,  2.10it/s]
[2024-11-27T09:26:06.709563+0000][460][CRITICAL] module disabled: /usr/local/lib/python3.10/dist-packages/synthcity/plugins/generic/plugin_goggle.py
  6%|▌         | 599/10000 [04:45<1:14:46,  2.10it/s]
[2024-11-27T09:36:46.247332+0000][460][CRITICAL] module disabled: /usr/local/lib/python3.10/dist-packages/synthcity/plugins/generic/plugin_goggle.py
[2024-11-27T09:36:46.270793+0000][460][CRITICAL] module disabled: /usr/local/lib/python3.10/dist-packages/synthcity/plugins/generic/plugin_goggle.py
[2024-11-27T09:36:46.292657+0000][460][CRITICAL] module disabled: /usr/local/lib/python3.10/dist-packages/synthcity/plugins/generic/plugin_goggle.py


An error  with adsgan: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Running image_adsgan. . . 
An error  with image_adsgan: Invalid dataloader type for image generators
Running image_cgan. . . 
An error  with image_cgan: Invalid dataloader type for image generators
Running uniform_sampler. . . 
